# CineInsight MovieLens 20M: profiling and analytics

This notebook accompanies the local medallion pipeline. Run `python src/ingestion/profiling.py`, then `python src/pipeline.py run`; the analysis reads the resulting profile and Gold marts. Rating key ordering and duplicates are checked across the full snapshot when the key ordering permits an exact adjacent-key check.


In [ ]:
from pathlib import Path
import json, sqlite3, pandas as pd
ROOT = Path.cwd()
profile = json.loads((ROOT/'lakehouse/reports/profile.json').read_text(encoding='utf-8'))
pd.DataFrame([{'file':x['file'],'rows':x['rows'],'empty_cells':x['empty_cells']} for x in profile])

## Data risks and design choices

The rating table drives the largest scan and should be partitioned by event month with clustering on movie/user. Do not partition on user ID. Rating values are expected on half-star increments in [0.5, 5]; release year is a parsed terminal suffix and can be absent; `(no genres listed)` is an explicit sentinel. Tags are user-entered free text and require trimming/case normalization before aggregation. Genome relevance is derived model output, not a user tag. The full `(userId,movieId)` ordering and duplicate check is exact when the file is nondecreasing by that key; repeated ratings at different timestamps remain valid events.


In [ ]:
for item in profile:
    print(item['file'], item['rows'])
    if 'rating_distribution' in item:
        print('rating values:', item['rating_distribution'])
        print('rating key order nondecreasing:', item['user_movie_key_order_nondecreasing'])
        print('duplicate user/movie pairs:', item['duplicate_user_movie_pairs'])
        print('event range:', item['event_time_min'], 'to', item['event_time_max'])
    if 'year_suffix_distribution' in item:
        print('year suffix distribution:', item['year_suffix_distribution'])

## Gold marts

The SQL outputs use a minimum 100-rating threshold for ranking. Without it, titles with one or two ratings can float to the top on chance. Genre variance uses population variance of rating values. Tag results include point-biserial (Pearson) correlation between a movie's normalized tag presence and its mean rating, using one observation per rated movie; the statistic is descriptive, not causal. The time charts are descriptive, not causal. Hidden gems are defined as a mean rating ≥4.0 with 10–99 ratings and an active, non-deleted movie. IMDb IDs are restored to seven digits in their URLs.


In [ ]:
reports = ROOT/'lakehouse/reports'
for name in ['top_movies.csv','genre_summary.csv','rating_by_year.csv','rating_by_event_month.csv','popular_tags.csv','tag_rating_association.csv','hidden_gems.csv','genome_coverage.csv','genome_action_group.csv']:
    path = reports/name
    if path.exists():
        print('\n', name)
        display(pd.read_csv(path).head(15))
    else:
        print(name, 'not generated yet; run src/pipeline.py run')

## Genome coverage and selected group

Coverage is the share of active catalog movies with at least one row in `fact_genome`. `genome_action_group.csv` summarizes the highest mean-relevance descriptors across active Action movies and counts the movies supporting each descriptor. Genome scores are algorithmically derived and should not be conflated with raw user tags.


In [ ]:
db = ROOT/'control/cineinsight.db'
if db.exists():
    with sqlite3.connect(db) as con:
        active = con.execute('select count(*) from dim_movie where is_current=1 and is_deleted=0').fetchone()[0]
        covered = con.execute('select count(distinct movie_id) from fact_genome').fetchone()[0]
        print({'active_movies': active, 'genome_covered_movies': covered, 'coverage_pct': round(100*covered/active,2) if active else None})
else:
    print('Run the pipeline to calculate genome coverage.')